# 03 Encode - Sample Or Full Memmap Embeddings

Sample mode writes embeddings under `data/embeddings/sample_<N>_per_category/`. Full mode writes the server-scale embeddings under `data/embeddings/`. Transformer outputs are resumable through `.meta.json` sidecars.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

sys.path.append('..')
from src.full_pipeline import (
    PROCESSED_PATH,
    EMBEDDING_DIR,
    parquet_row_count,
    sample_processed_path,
    sample_embedding_dir,
    write_all_embeddings,
)


## 1. Encoding Configuration


In [ ]:
RUN_MODE = "sample"  # "sample" for local, "full" for server
ROWS_PER_CATEGORY = 1_000

RUN_TFIDF = True
RUN_W2V = True
RUN_GLOVE = True if RUN_MODE == "sample" else False  # keep off by default for 16GB full runs
RUN_SBERT = True
RUN_BGE = True
BGE_MODEL_NAME = "BAAI/bge-large-en-v1.5"

FORCE_REBUILD = False
TEXT_BATCH_SIZE = 4096 if RUN_MODE == "sample" else 8192
TRANSFORMER_BATCH_SIZE = 16 if (RUN_MODE == "sample" and not torch.cuda.is_available()) else (64 if not torch.cuda.is_available() else 256)

ACTIVE_PROCESSED_PATH = sample_processed_path(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else PROCESSED_PATH
ACTIVE_EMBEDDING_DIR = sample_embedding_dir(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else EMBEDDING_DIR
MIN_EXPECTED_ROWS = 100 if RUN_MODE == "sample" else 1_000_000

if not ACTIVE_PROCESSED_PATH.exists():
    raise FileNotFoundError(f"{ACTIVE_PROCESSED_PATH} not found. Run 02_preprocess.ipynb first with the same RUN_MODE/ROWS_PER_CATEGORY.")

row_count = parquet_row_count(ACTIVE_PROCESSED_PATH)
if row_count < MIN_EXPECTED_ROWS:
    raise RuntimeError(f"{ACTIVE_PROCESSED_PATH} has only {row_count:,} rows; rerun 02_preprocess.ipynb.")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Run mode: {RUN_MODE}")
print(f"Processed rows: {row_count:,}")
print(f"Embedding dir: {ACTIVE_EMBEDDING_DIR}")
print(f"Torch device: {device}")
print(f"Transformer batch size: {TRANSFORMER_BATCH_SIZE}")
print(f"BGE model: {BGE_MODEL_NAME}")
if device == 'cpu' and RUN_BGE:
    print("Warning: BGE-large on CPU is slow; this sample smoke run may take hours locally.")


## 2. Run Encoders


In [ ]:
embedding_summary = write_all_embeddings(
    processed_path=ACTIVE_PROCESSED_PATH,
    embedding_dir=ACTIVE_EMBEDDING_DIR,
    run_tfidf=RUN_TFIDF,
    run_w2v=RUN_W2V,
    run_glove=RUN_GLOVE,
    run_sbert=RUN_SBERT,
    run_bge=RUN_BGE,
    bge_model_name=BGE_MODEL_NAME,
    force=FORCE_REBUILD,
    transformer_batch_size=TRANSFORMER_BATCH_SIZE,
    text_batch_size=TEXT_BATCH_SIZE,
)

display(pd.DataFrame.from_dict(embedding_summary, orient='index'))


## 3. Inspect Saved Embedding Shapes


In [ ]:
rows = []
for path in sorted(ACTIVE_EMBEDDING_DIR.glob('*.npy')):
    arr = np.load(path, mmap_mode='r')
    rows.append({'encoder': path.stem, 'shape': arr.shape, 'dtype': str(arr.dtype), 'size_gb': path.stat().st_size / 1e9})

display(pd.DataFrame(rows))
